In [1]:
! python --version

In [2]:
import sys
print(sys.executable)


In [10]:
import math
import requests

# ------------------------------------------------------
# Step 1. Fetch required weather data from Open-Meteo
# ------------------------------------------------------

url = (
    "https://api.open-meteo.com/v1/forecast?"
    "latitude=6.9271&longitude=79.8612"
    "&daily=temperature_2m_max,temperature_2m_min,"
    "relative_humidity_2m_max,relative_humidity_2m_min,"
    "wind_speed_10m_max,shortwave_radiation_sum"
    "&timezone=Asia/Colombo"
)

response = requests.get(url)
daily = response.json()["daily"]

Tmax = daily["temperature_2m_max"][0]
Tmin = daily["temperature_2m_min"][0]
RHmax = daily["relative_humidity_2m_max"][0]
RHmin = daily["relative_humidity_2m_min"][0]
wind_speed = daily["wind_speed_10m_max"][0]
solar_radiation = daily["shortwave_radiation_sum"][0]  # MJ/m2 per day

# ------------------------------------------------------
# Step 2. Helper functions for Penman Monteith
# ------------------------------------------------------

def saturation_vapor_pressure(T):
    return 0.6108 * math.exp((17.27 * T) / (T + 237.3))

def slope_svp_curve(T):
    return 4098 * (0.6108 * math.exp((17.27 * T) / (T + 237.3))) / ((T + 237.3) ** 2)

def actual_vapor_pressure(RHmin, RHmax, es_min, es_max):
    ea = (RHmax / 100) * es_min + (RHmin / 100) * es_max
    return ea / 2

def psychrometric_constant():
    # Colombo elevation around 10 m
    P = 101.3  # kPa
    gamma = 0.000665 * P
    return gamma

# ------------------------------------------------------
# Step 3. Compute intermediate variables
# ------------------------------------------------------

Tmean = (Tmax + Tmin) / 2

es_max = saturation_vapor_pressure(Tmax)
es_min = saturation_vapor_pressure(Tmin)
es = (es_max + es_min) / 2

ea = actual_vapor_pressure(RHmin, RHmax, es_min, es_max)

delta = slope_svp_curve(Tmean)
gamma = psychrometric_constant()

Rn = solar_radiation  # already MJ/m2 per day
G = 0  # soil heat flux for daily step

u2 = wind_speed  # already 2 metre equivalent

# ------------------------------------------------------
# Step 4. Penman Monteith Equation
# ------------------------------------------------------

ET0 = (
    0.408 * delta * (Rn - G)
    + gamma * (900 / (Tmean + 273)) * u2 * (es - ea)
) / (delta + gamma * (1 + 0.34 * u2))

print("Daily ET0 (mm per day):", ET0)


perplexity

In [14]:
import math
import requests

# ------------------------------------------------------
# Step 1. Fetch required weather data from Open-Meteo
# ------------------------------------------------------
# Note: we now request wind_speed_10m_mean instead of max.

url = (
    "https://api.open-meteo.com/v1/forecast?"
    "latitude=6.9271&longitude=79.8612"
    "&daily=temperature_2m_max,temperature_2m_min,"
    "relative_humidity_2m_max,relative_humidity_2m_min,"
    "wind_speed_10m_mean,shortwave_radiation_sum"
    "&timezone=Asia/Colombo"
)

response = requests.get(url)
daily = response.json()["daily"]

Tmax = daily["temperature_2m_max"][0]
Tmin = daily["temperature_2m_min"][0]
RHmax = daily["relative_humidity_2m_max"][0]
RHmin = daily["relative_humidity_2m_min"][0]
wind_speed_10m_mean = daily["wind_speed_10m_mean"][0]   # m/s at 10 m
solar_radiation = daily["shortwave_radiation_sum"][0]   # MJ/m² per day

# ------------------------------------------------------
# Step 2. Helper functions for Penman-Monteith
# ------------------------------------------------------

def saturation_vapor_pressure(T):
    """Saturation vapour pressure e_s(T) in kPa for temperature T in °C."""
    return 0.6108 * math.exp((17.27 * T) / (T + 237.3))

def slope_svp_curve(T):
    """Slope of saturation vapour pressure curve Delta in kPa/°C."""
    es_T = saturation_vapor_pressure(T)
    return 4098.0 * es_T / ((T + 237.3) ** 2)

def actual_vapor_pressure(RHmin, RHmax, es_min, es_max):
    """
    Actual vapour pressure ea in kPa.
    FAO approximation using min/max RH and corresponding saturation vapour pressures.
    """
    ea = (RHmax / 100.0) * es_min + (RHmin / 100.0) * es_max
    return ea / 2.0

def psychrometric_constant(elevation_m=10.0):
    """
    Psychrometric constant gamma in kPa/°C.
    Pressure P (kPa) approximated from elevation.
    """
    P = 101.3 * ((293.0 - 0.0065 * elevation_m) / 293.0) ** 5.26
    gamma = 0.000665 * P
    return gamma

def wind_speed_10m_to_2m(u10):
    """
    Convert wind speed from 10 m height to 2 m height using FAO-56 log wind profile:
    u2 = uz * 4.87 / ln(67.8*z - 5.42)
    """
    z = 10.0
    return u10 * 4.87 / math.log(67.8 * z - 5.42)

def simple_net_radiation(Rs, Tmax, Tmin, ea):
    """
    Approximate net radiation Rn (MJ/m²/day) from shortwave radiation Rs,
    using FAO-56 style components (simplified):
      Rns = (1 - albedo) * Rs, albedo ~ 0.23
      Rnl = simplified net longwave radiation term.
    """
    albedo = 0.23
    Rns = (1.0 - albedo) * Rs  # net shortwave

    sigma = 4.903e-9  # Stefan-Boltzmann constant (MJ K^-4 m^-2 day^-1)
    Tmax_K = Tmax + 273.16
    Tmin_K = Tmin + 273.16
    Tavg4 = (Tmax_K**4 + Tmin_K**4) / 2.0

    # Emissivity and cloud factor (simplified)
    emissivity = 0.34 - 0.14 * math.sqrt(max(ea, 0.0))
    cloud_factor = 0.9  # rough average; ideally depends on Rs/Rso
    Rnl = sigma * Tavg4 * emissivity * cloud_factor

    Rn = Rns - Rnl
    return Rn

# ------------------------------------------------------
# Step 3. Compute intermediate variables
# ------------------------------------------------------

Tmean = (Tmax + Tmin) / 2.0

es_max = saturation_vapor_pressure(Tmax)
es_min = saturation_vapor_pressure(Tmin)
es = (es_max + es_min) / 2.0

ea = actual_vapor_pressure(RHmin, RHmax, es_min, es_max)

delta = slope_svp_curve(Tmean)
gamma = psychrometric_constant(elevation_m=10.0)

# Net radiation (better than using Rs directly)
Rn = simple_net_radiation(solar_radiation, Tmax, Tmin, ea)
G = 0.0  # soil heat flux for daily timestep

# Convert mean wind speed from 10 m to 2 m
u2 = wind_speed_10m_to_2m(wind_speed_10m_mean)

# ------------------------------------------------------
# Step 4. Penman-Monteith Equation (daily ET0)
# ------------------------------------------------------

ET0 = (
    0.408 * delta * (Rn - G)
    + gamma * (900.0 / (Tmean + 273.0)) * u2 * (es - ea)
) / (delta + gamma * (1.0 + 0.34 * u2))

print("Daily ET0 for Colombo (mm/day):", ET0)


gpt plus

In [13]:
import math
import requests

# ------------------------------------------------------
# Step 1. Fetch required weather data from Open Meteo
# ------------------------------------------------------

url = (
    "https://api.open-meteo.com/v1/forecast?"
    "latitude=6.9271&longitude=79.8612"
    "&daily=temperature_2m_max,temperature_2m_min,"
    "relative_humidity_2m_max,relative_humidity_2m_min,"
    "wind_speed_10m_mean,shortwave_radiation_sum,"
    "sunshine_duration"
    "&timezone=Asia/Colombo"
)

response = requests.get(url)
daily = response.json()["daily"]

Tmax = daily["temperature_2m_max"][0]
Tmin = daily["temperature_2m_min"][0]
RHmax = daily["relative_humidity_2m_max"][0]
RHmin = daily["relative_humidity_2m_min"][0]
wind_10m_mean = daily["wind_speed_10m_mean"][0]
Rs = daily["shortwave_radiation_sum"][0]          # MJ m^-2 day^-1
sunshine = daily["sunshine_duration"][0] / 3600   # convert seconds to hours


# ------------------------------------------------------
# Step 2. FAO 56 helper functions
# ------------------------------------------------------

def saturation_vapor_pressure(T):
    return 0.6108 * math.exp((17.27 * T) / (T + 237.3))

def slope_svp_curve(T):
    es = saturation_vapor_pressure(T)
    return (4098 * es) / ((T + 237.3) ** 2)

def actual_vapor_pressure(RHmin, RHmax, es_min, es_max):
    ea = (RHmax * es_min + RHmin * es_max) / 200
    return ea

def psychrometric_constant(P=101.3):
    return 0.000665 * P

def wind_10m_to_2m(u10):
    return u10 * (4.87 / math.log(67.8 * 10 - 5.42))

def clear_sky_radiation(Ra, elevation=10):
    return (0.75 + 2e-5 * elevation) * Ra

def net_shortwave_radiation(Rs):
    albedo = 0.23
    return (1 - albedo) * Rs

def net_longwave_radiation(Tmax, Tmin, ea, Rs, Rso):
    TmaxK = Tmax + 273.16
    TminK = Tmin + 273.16
    sigma = 4.903e-9

    term1 = (TmaxK ** 4 + TminK ** 4) / 2
    term2 = 0.34 - 0.14 * math.sqrt(ea)
    term3 = 1.35 * (Rs / Rso) - 0.35

    return sigma * term1 * term2 * term3


# ------------------------------------------------------
# Step 3. Compute daily variables
# ------------------------------------------------------

Tmean = (Tmax + Tmin) / 2

es_max = saturation_vapor_pressure(Tmax)
es_min = saturation_vapor_pressure(Tmin)
es = (es_max + es_min) / 2

ea = actual_vapor_pressure(RHmin, RHmax, es_min, es_max)

delta = slope_svp_curve(Tmean)
gamma = psychrometric_constant()

u2 = wind_10m_to_2m(wind_10m_mean)   # corrected wind speed


# ------------------------------------------------------
# Step 4. Safe and stable radiation calculations
# ------------------------------------------------------

if sunshine <= 0:
    sunshine = 0.1

Ra = max(0.1, 15.392 * sunshine)

Rso = clear_sky_radiation(Ra)
Rso = max(Rso, 0.1)

Rns = net_shortwave_radiation(Rs)
Rnl = net_longwave_radiation(Tmax, Tmin, ea, Rs, Rso)

Rn = Rns - Rnl
G = 0


# ------------------------------------------------------
# Step 5. Penman Monteith ET0
# ------------------------------------------------------

ET0 = (
    0.408 * delta * (Rn - G)
    + gamma * (900 / (Tmean + 273)) * u2 * (es - ea)
) / (delta + gamma * (1 + 0.34 * u2))

print("Daily ET0 (mm per day):", ET0)


gpt to perplexity

In [ ]:
import math

# ================================
# Helper functions
# ================================

def saturation_vapor_pressure(T):
    """Saturation vapour pressure e_s(T) in kPa for temperature T in Celsius."""
    return 0.6108 * math.exp((17.27 * T) / (T + 237.3))

def slope_svp_curve(T):
    """Slope of saturation vapour pressure curve Delta in kPa/°C."""
    es_T = saturation_vapor_pressure(T)
    return 4098.0 * es_T / ((T + 237.3) ** 2)

def actual_vapor_pressure(RHmin, RHmax, es_min, es_max):
    """
    Actual vapour pressure ea in kPa.
    FAO approximation using min and max relative humidity and min/max saturation vapour pressure.
    """
    ea = (RHmax / 100.0) * es_min + (RHmin / 100.0) * es_max
    return ea / 2.0

def psychrometric_constant(elevation_m=10.0):
    """
    Psychrometric constant gamma in kPa/°C.
    Pressure P (kPa) approximated from elevation.
    """
    P = 101.3 * ((293.0 - 0.0065 * elevation_m) / 293.0) ** 5.26
    gamma = 0.000665 * P
    return gamma

def wind_speed_10m_to_2m(u10):
    """
    Convert wind speed from 10 m height to 2 m height using FAO-56 log wind profile.
    u2 = uz * 4.87 / ln(67.8*z - 5.42), for z in m.
    """
    z = 10.0
    u2 = u10 * 4.87 / math.log(67.8 * z - 5.42)
    return u2

# ================================
# Simple net radiation
# ================================

def simple_net_radiation(Rs, Tmax, Tmin, ea):
    """
    Approximate net radiation Rn (MJ/m²/day) from shortwave radiation Rs,
    using FAO-56 style components:
      Rns = (1 - albedo) * Rs, albedo ~ 0.23
      Rnl = simplified net longwave radiation term.
    This is a simplification; for full FAO-56, you need Rso and cloudiness.
    """
    albedo = 0.23
    Rns = (1.0 - albedo) * Rs  # net shortwave

    sigma = 4.903e-9  # Stefan-Boltzmann constant (MJ K^-4 m^-2 day^-1)
    Tmax_K = Tmax + 273.16
    Tmin_K = Tmin + 273.16
    Tavg4 = (Tmax_K**4 + Tmin_K**4) / 2.0

    # Emissivity and cloud factor (simplified)
    emissivity = 0.34 - 0.14 * math.sqrt(max(ea, 0.0))
    cloud_factor = 0.9  # rough average; ideally depends on Rs/Rso
    Rnl = sigma * Tavg4 * emissivity * cloud_factor

    Rn = Rns - Rnl
    return Rn

# ================================
# Main ET0 function
# ================================

def compute_daily_et0_row(row, elevation_m=10.0):
    """
    Compute ET0 for a single day using the FAO Penman-Monteith formula.

    Expects a row with:
      temperature_2m_max, temperature_2m_min,
      relative_humidity_2m_max, relative_humidity_2m_min,
      wind_speed_10m_mean, shortwave_radiation_sum (MJ/m²/day)
    """
    Tmax = row["temperature_2m_max"]
    Tmin = row["temperature_2m_min"]
    RHmax = row["relative_humidity_2m_max"]
    RHmin = row["relative_humidity_2m_min"]

    # Use daily mean wind speed at 10 m
    u10_mean = row["wind_speed_10m_mean"]  # m/s at 10 m
    Rs = row["shortwave_radiation_sum"]    # MJ/m² per day

    Tmean = (Tmax + Tmin) / 2.0

    # Saturation vapour pressures
    es_max = saturation_vapor_pressure(Tmax)
    es_min = saturation_vapor_pressure(Tmin)
    es = (es_max + es_min) / 2.0

    # Actual vapour pressure
    ea = actual_vapor_pressure(RHmin, RHmax, es_min, es_max)

    # Slope of SVP curve
    delta = slope_svp_curve(Tmean)

    # Psychrometric constant
    gamma = psychrometric_constant(elevation_m)

    # Net radiation (improved)
    Rn = simple_net_radiation(Rs, Tmax, Tmin, ea)
    G = 0.0  # soil heat flux for daily step

    # Convert wind speed from 10 m to 2 m
    u2 = wind_speed_10m_to_2m(u10_mean)

    # FAO Penman-Monteith equation (daily ET0, mm/day)
    numerator = 0.408 * delta * (Rn - G) + gamma * (900.0 / (Tmean + 273.0)) * u2 * (es - ea)
    denominator = delta + gamma * (1.0 + 0.34 * u2)
    ET0 = numerator / denominator

    return ET0


In [16]:
"""
FAO Penman-Monteith Evapotranspiration Calculator
Using Open-Meteo API for Colombo, Sri Lanka
Date: November 26, 2025
"""

import requests
import math
from datetime import datetime

# ============================================================================
# LOCATION PARAMETERS
# ============================================================================
latitude = 6.9271      # Colombo latitude
longitude = 79.8612    # Colombo longitude
elevation = 7          # meters above sea level
today = "2025-11-26"

# ============================================================================
# STEP 1: FETCH WEATHER DATA FROM OPEN-METEO API
# ============================================================================
def fetch_weather_data(lat, lon, date):
    """Fetch weather data from Open-Meteo API"""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ["temperature_2m", "relative_humidity_2m", 
                   "wind_speed_10m", "shortwave_radiation"],
        "daily": ["temperature_2m_max", "temperature_2m_min"],
        "timezone": "Asia/Colombo",
        "start_date": date,
        "end_date": date
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    return data

# ============================================================================
# STEP 2: CALCULATE MEAN VALUES FROM HOURLY DATA
# ============================================================================
def calculate_daily_means(hourly_data):
    """Calculate daily mean values from hourly data"""
    temp = hourly_data['temperature_2m']
    rh = hourly_data['relative_humidity_2m']
    wind = hourly_data['wind_speed_10m']
    radiation = hourly_data['shortwave_radiation']
    
    # Filter out None values and calculate means
    temp_mean = sum(t for t in temp if t is not None) / len([t for t in temp if t is not None])
    rh_mean = sum(r for r in rh if r is not None) / len([r for r in rh if r is not None])
    wind_mean = sum(w for w in wind if w is not None) / len([w for w in wind if w is not None])
    
    # Convert shortwave radiation (W/m²) to MJ/m²/day
    radiation_sum = sum(r for r in radiation if r is not None)
    solar_radiation = (radiation_sum * 3600) / 1000000  # MJ/m²/day
    
    return temp_mean, rh_mean, wind_mean, solar_radiation

# ============================================================================
# STEP 3: FAO PENMAN-MONTEITH EQUATION COMPONENTS
# ============================================================================
def saturation_vapour_pressure(T):
    """Calculate saturation vapour pressure (kPa) at temperature T (°C)"""
    return 0.6108 * math.exp((17.27 * T) / (T + 237.3))

def slope_vapour_pressure(T):
    """Calculate slope of saturation vapour pressure curve (kPa/°C)"""
    return (4098 * saturation_vapour_pressure(T)) / ((T + 237.3) ** 2)

def atmospheric_pressure(altitude):
    """Calculate atmospheric pressure (kPa) based on altitude (m)"""
    return 101.3 * ((293 - 0.0065 * altitude) / 293) ** 5.26

def psychrometric_constant(P):
    """Calculate psychrometric constant (kPa/°C)"""
    return 0.000665 * P

def actual_vapour_pressure(T, RH):
    """Calculate actual vapour pressure (kPa) from temperature and relative humidity"""
    es = saturation_vapour_pressure(T)
    return (RH / 100.0) * es

def net_radiation(Rs, T_max, T_min, ea, latitude, day_of_year):
    """
    Calculate net radiation (MJ/m²/day)
    Rs: Solar radiation (MJ/m²/day)
    """
    # Albedo for grass reference crop
    albedo = 0.23
    
    # Net shortwave radiation
    Rns = (1 - albedo) * Rs
    
    # Stefan-Boltzmann constant (MJ/K⁴/m²/day)
    sigma = 4.903e-9
    
    # Net longwave radiation
    T_max_K = T_max + 273.16
    T_min_K = T_min + 273.16
    
    # Clear sky radiation (approximation)
    Ra = extraterrestrial_radiation(latitude, day_of_year)
    Rso = (0.75 + 2e-5 * elevation) * Ra
    
    # Net longwave radiation
    Rnl = sigma * ((T_max_K**4 + T_min_K**4) / 2) * (0.34 - 0.14 * math.sqrt(ea)) * ((1.35 * Rs / Rso) - 0.35)
    
    # Net radiation
    Rn = Rns - Rnl
    return Rn

def extraterrestrial_radiation(lat, day_of_year):
    """Calculate extraterrestrial radiation Ra (MJ/m²/day)"""
    Gsc = 0.0820  # Solar constant (MJ/m²/min)
    lat_rad = (math.pi / 180) * lat
    dr = 1 + 0.033 * math.cos((2 * math.pi / 365) * day_of_year)
    delta = 0.409 * math.sin((2 * math.pi / 365) * day_of_year - 1.39)
    ws = math.acos(-math.tan(lat_rad) * math.tan(delta))
    
    Ra = (24 * 60 / math.pi) * Gsc * dr * (
        ws * math.sin(lat_rad) * math.sin(delta) + 
        math.cos(lat_rad) * math.cos(delta) * math.sin(ws)
    )
    return Ra

def wind_speed_2m(u_z, z=10):
    """Convert wind speed at height z (m) to wind speed at 2m height"""
    return u_z * (4.87 / math.log(67.8 * z - 5.42))

# ============================================================================
# STEP 4: CALCULATE REFERENCE EVAPOTRANSPIRATION (ET0)
# ============================================================================
def calculate_ET0(T_max, T_min, RH, u_2, Rs, altitude, latitude, day_of_year):
    """
    Calculate reference evapotranspiration (ET0) using FAO Penman-Monteith equation
    
    Parameters:
    - T_max: Maximum temperature (°C)
    - T_min: Minimum temperature (°C)
    - RH: Relative humidity (%)
    - u_2: Wind speed at 2m height (m/s)
    - Rs: Solar radiation (MJ/m²/day)
    - altitude: Elevation (m)
    - latitude: Latitude (degrees)
    - day_of_year: Day of year (1-365)
    
    Returns:
    - ET0: Reference evapotranspiration (mm/day)
    """
    
    # Mean temperature
    T_mean = (T_max + T_min) / 2
    
    # Saturation vapour pressure
    es = (saturation_vapour_pressure(T_max) + saturation_vapour_pressure(T_min)) / 2
    
    # Actual vapour pressure
    ea = actual_vapour_pressure(T_mean, RH)
    
    # Slope of saturation vapour pressure curve
    Delta = slope_vapour_pressure(T_mean)
    
    # Atmospheric pressure
    P = atmospheric_pressure(altitude)
    
    # Psychrometric constant
    gamma = psychrometric_constant(P)
    
    # Net radiation
    Rn = net_radiation(Rs, T_max, T_min, ea, latitude, day_of_year)
    
    # Soil heat flux (assumed 0 for daily calculations)
    G = 0
    
    # FAO Penman-Monteith equation
    numerator = 0.408 * Delta * (Rn - G) + gamma * (900 / (T_mean + 273)) * u_2 * (es - ea)
    denominator = Delta + gamma * (1 + 0.34 * u_2)
    
    ET0 = numerator / denominator
    
    return ET0, {
        'T_mean': T_mean,
        'es': es,
        'ea': ea,
        'Delta': Delta,
        'P': P,
        'gamma': gamma,
        'Rn': Rn,
        'VPD': es - ea
    }

# ============================================================================
# MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    print("=" * 70)
    print("FAO PENMAN-MONTEITH EVAPOTRANSPIRATION CALCULATOR")
    print("=" * 70)
    print(f"Location: Colombo, Sri Lanka")
    print(f"Coordinates: {latitude}°N, {longitude}°E")
    print(f"Elevation: {elevation} m")
    print(f"Date: {today}")
    print("=" * 70)
    
    try:
        # Fetch weather data
        print("\nFetching weather data from Open-Meteo API...")
        weather_data = fetch_weather_data(latitude, longitude, today)
        
        # Extract daily values
        T_max = weather_data['daily']['temperature_2m_max'][0]
        T_min = weather_data['daily']['temperature_2m_min'][0]
        
        # Calculate means from hourly data
        T_mean, RH_mean, wind_mean, Rs = calculate_daily_means(weather_data['hourly'])
        
        # Convert wind speed from 10m to 2m
        u_2 = wind_speed_2m(wind_mean, z=10)
        
        # Get day of year
        date_obj = datetime.strptime(today, "%Y-%m-%d")
        day_of_year = date_obj.timetuple().tm_yday
        
        print("\nWeather Data Retrieved:")
        print(f"  Maximum Temperature: {T_max:.1f}°C")
        print(f"  Minimum Temperature: {T_min:.1f}°C")
        print(f"  Mean Temperature: {T_mean:.1f}°C")
        print(f"  Mean Relative Humidity: {RH_mean:.1f}%")
        print(f"  Mean Wind Speed (10m): {wind_mean:.2f} m/s")
        print(f"  Wind Speed (2m): {u_2:.2f} m/s")
        print(f"  Solar Radiation: {Rs:.2f} MJ/m²/day")
        
        # Calculate ET0
        print("\nCalculating Reference Evapotranspiration (ET0)...")
        ET0, params = calculate_ET0(T_max, T_min, RH_mean, u_2, Rs, 
                                     elevation, latitude, day_of_year)
        
        print("\n" + "=" * 70)
        print("RESULTS")
        print("=" * 70)
        print(f"Reference Evapotranspiration (ET0): {ET0:.2f} mm/day")
        print("\nIntermediate Parameters:")
        print(f"  Mean Temperature: {params['T_mean']:.2f}°C")
        print(f"  Saturation Vapour Pressure (es): {params['es']:.3f} kPa")
        print(f"  Actual Vapour Pressure (ea): {params['ea']:.3f} kPa")
        print(f"  Vapour Pressure Deficit (VPD): {params['VPD']:.3f} kPa")
        print(f"  Slope of SVP curve (Δ): {params['Delta']:.4f} kPa/°C")
        print(f"  Atmospheric Pressure (P): {params['P']:.2f} kPa")
        print(f"  Psychrometric Constant (γ): {params['gamma']:.4f} kPa/°C")
        print(f"  Net Radiation (Rn): {params['Rn']:.2f} MJ/m²/day")
        print("=" * 70)
        
    except Exception as e:
        print(f"\nError: {e}")
        print("\nPlease check your internet connection and try again.")


In [20]:
import math
import requests
from datetime import date

# ================================
# 0. Site and date
# ================================
LATITUDE = 6.94      # Colombo, Sri Lanka [deg] [web:79]
LONGITUDE = 79.85    # [deg] [web:79]
ELEVATION_M = 11.0   # m above sea level (approx.) [web:81]

today = date.today().isoformat()
day_of_year = date.today().timetuple().tm_yday

# ================================
# 1. Helper functions (FAO-56)
# ================================

def saturation_vapor_pressure(T):
    # e_s(T) in kPa
    return 0.6108 * math.exp((17.27 * T) / (T + 237.3))

def slope_svp_curve(T):
    es_T = saturation_vapor_pressure(T)
    return 4098.0 * es_T / ((T + 237.3) ** 2)

def actual_vapor_pressure(RHmin, RHmax, es_min, es_max):
    ea = (RHmax / 100.0) * es_min + (RHmin / 100.0) * es_max
    return ea / 2.0

def psychrometric_constant(elevation_m):
    P = 101.3 * ((293.0 - 0.0065 * elevation_m) / 293.0) ** 5.26
    gamma = 0.000665 * P
    return gamma

def wind_speed_10m_to_2m(u10):
    z = 10.0
    return u10 * 4.87 / math.log(67.8 * z - 5.42)

def extraterrestrial_radiation(day_of_year, latitude_deg):
    phi = math.radians(latitude_deg)
    dr = 1 + 0.033 * math.cos((2 * math.pi / 365) * day_of_year)
    delta = 0.409 * math.sin((2 * math.pi / 365) * day_of_year - 1.39)
    ws = math.acos(-math.tan(phi) * math.tan(delta))
    Gsc = 0.0820  # MJ m-2 min-1

    Ra = (24 * 60 / math.pi) * Gsc * dr * (
        ws * math.sin(phi) * math.sin(delta) +
        math.cos(phi) * math.cos(delta) * math.sin(ws)
    )
    return Ra

def clear_sky_radiation(Ra, elevation_m):
    return (0.75 + 2e-5 * elevation_m) * Ra

def net_radiation_fao(Rs, Tmax, Tmin, ea, Ra, elevation_m):
    # Net shortwave
    albedo = 0.23
    Rns = (1.0 - albedo) * Rs

    # Net longwave
    sigma = 4.903e-9  # MJ K^-4 m^-2 day^-1
    Tmax_K = Tmax + 273.16
    Tmin_K = Tmin + 273.16
    Tavg4 = (Tmax_K**4 + Tmin_K**4) / 2.0

    Rso = clear_sky_radiation(Ra, elevation_m)
    Rs_Rso = Rs / Rso if Rso > 0 else 0.0
    Rs_Rso = max(0.0, min(Rs_Rso, 1.5))

    Rnl = sigma * Tavg4 * (0.34 - 0.14 * math.sqrt(max(ea, 0.0))) * (1.35 * Rs_Rso - 0.35)
    return Rns - Rnl

def compute_daily_et0(Tmax, Tmin, RHmax, RHmin, u10_mean, Rs,
                      elevation_m, latitude_deg, day_of_year):
    Tmean = (Tmax + Tmin) / 2.0

    # Saturation vapour pressures
    es_max = saturation_vapor_pressure(Tmax)
    es_min = saturation_vapor_pressure(Tmin)
    es = (es_max + es_min) / 2.0

    # Actual vapour pressure
    ea = actual_vapor_pressure(RHmin, RHmax, es_min, es_max)

    # Slope of SVP curve
    delta = slope_svp_curve(Tmean)

    # Psychrometric constant
    gamma = psychrometric_constant(elevation_m)

    # Radiation
    Ra = extraterrestrial_radiation(day_of_year, latitude_deg)
    Rn = net_radiation_fao(Rs, Tmax, Tmin, ea, Ra, elevation_m)

    G = 0.0  # daily soil heat flux
    u2 = wind_speed_10m_to_2m(u10_mean)

    numerator = (0.408 * delta * (Rn - G) +
                 gamma * (900.0 / (Tmean + 273.0)) * u2 * (es - ea))
    denominator = delta + gamma * (1.0 + 0.34 * u2)
    ET0 = numerator / max(denominator, 1e-6)
    return ET0

# ================================
# 2. Fetch Open-Meteo daily data
# ================================

open_meteo_url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "relative_humidity_2m_max",
        "relative_humidity_2m_min",
        "wind_speed_10m_mean",
        "shortwave_radiation_sum",
    ],
    "timezone": "auto",
    "start_date": today,
    "end_date": today,
}

resp = requests.get(open_meteo_url, params=params)
resp.raise_for_status()
daily = resp.json()["daily"]  # Open-Meteo daily block [web:73]

# Extract today’s values (index 0)
Tmax = daily["temperature_2m_max"][0]
Tmin = daily["temperature_2m_min"][0]
RHmax = daily["relative_humidity_2m_max"][0]
RHmin = daily["relative_humidity_2m_min"][0]
u10_mean = daily["wind_speed_10m_mean"][0]
Rs = daily["shortwave_radiation_sum"][0]  # MJ/m²/day [web:73]

print("Open-Meteo inputs for today in Colombo:")
print(f"Tmax = {Tmax} °C, Tmin = {Tmin} °C")
print(f"RHmax = {RHmax} %, RHmin = {RHmin} %")
print(f"Wind speed (10 m mean) = {u10_mean} m/s")
print(f"Shortwave radiation sum = {Rs} MJ/m²/day")

# ================================
# 3. Compute ET0 (FAO-56 PM)
# ================================

et0_today = compute_daily_et0(
    Tmax=Tmax,
    Tmin=Tmin,
    RHmax=RHmax,
    RHmin=RHmin,
    u10_mean=u10_mean,
    Rs=Rs,
    elevation_m=ELEVATION_M,
    latitude_deg=LATITUDE,
    day_of_year=day_of_year,
)

print(f"\nReference ET0 (FAO-56 PM) for Colombo today: {et0_today:.3f} mm/day")


In [21]:
import requests
from datetime import date

# Colombo coordinates
lat = 6.9271
lon = 79.8612

url = (
    f"https://api.open-meteo.com/v1/forecast?"
    f"latitude={lat}&longitude={lon}"
    f"&daily=et0_fao_evapotranspiration"
    f"&timezone=Asia%2FColombo"
)

response = requests.get(url)
data = response.json()

dates = data["daily"]["time"]
et0 = data["daily"]["et0_fao_evapotranspiration"]

today = str(date.today())

if today in dates:
    idx = dates.index(today)
    print(f"Today's ET0 (FAO-56) in Colombo: {et0[idx]} mm/day")
else:
    print("Today's ET0 not found in API response")


In [ ]:
import math

# ====================================================
# 4. Penman Monteith helper functions (FAO-56, daily)
# ====================================================

def saturation_vapor_pressure(T):
    """Saturation vapour pressure e_s(T) in kPa for temperature T in Celsius."""
    return 0.6108 * math.exp((17.27 * T) / (T + 237.3))

def slope_svp_curve(T):
    """Slope of saturation vapour pressure curve Delta in kPa/°C."""
    es_T = saturation_vapor_pressure(T)
    return 4098.0 * es_T / ((T + 237.3) ** 2)

def actual_vapor_pressure(RHmin, RHmax, es_min, es_max):
    """
    Actual vapour pressure ea in kPa.
    FAO approximation using min/max RH and corresponding saturation vapour pressures.
    """
    ea = (RHmax / 100.0) * es_min + (RHmin / 100.0) * es_max
    return ea / 2.0

def psychrometric_constant(elevation_m=10.0):
    """
    Psychrometric constant gamma in kPa/°C.
    Pressure P (kPa) approximated from elevation (FAO-56).
    """
    P = 101.3 * ((293.0 - 0.0065 * elevation_m) / 293.0) ** 5.26
    gamma = 0.000665 * P
    return gamma

def wind_speed_10m_to_2m(u10):
    """
    Convert wind speed from 10 m height to 2 m height using FAO-56 log wind profile.
    u2 = uz * 4.87 / ln(67.8*z - 5.42), with z in m.
    """
    z = 10.0
    return u10 * 4.87 / math.log(67.8 * z - 5.42)

def extraterrestrial_radiation(day_of_year, latitude_deg):
    """
    Extraterrestrial radiation Ra (MJ/m²/day) for daily periods (FAO-56 Eq. 21–25).
    latitude_deg: decimal degrees
    """
    phi = math.radians(latitude_deg)
    dr = 1 + 0.033 * math.cos((2 * math.pi / 365) * day_of_year)
    delta = 0.409 * math.sin((2 * math.pi / 365) * day_of_year - 1.39)
    ws = math.acos(-math.tan(phi) * math.tan(delta))
    Gsc = 0.0820  # MJ m-2 min-1

    Ra = (24 * 60 / math.pi) * Gsc * dr * (
        ws * math.sin(phi) * math.sin(delta) +
        math.cos(phi) * math.cos(delta) * math.sin(ws)
    )
    return Ra

def clear_sky_radiation(Ra, elevation_m):
    """
    Clear-sky solar radiation Rso (MJ/m²/day) (FAO-56 Eq. 37).
    """
    return (0.75 + 2e-5 * elevation_m) * Ra

def net_radiation_fao(Rs, Tmax, Tmin, ea, Ra, elevation_m):
    """
    Net radiation Rn (MJ/m²/day) using FAO-56:
      Rns = (1 - albedo) * Rs
      Rnl = σ * (TmaxK^4 + TminK^4)/2 * (0.34 - 0.14 * sqrt(ea)) * (1.35*Rs/Rso - 0.35)
    """
    # Net shortwave radiation
    albedo = 0.23
    Rns = (1.0 - albedo) * Rs

    # Net longwave radiation
    sigma = 4.903e-9  # MJ K^-4 m^-2 day^-1
    Tmax_K = Tmax + 273.16
    Tmin_K = Tmin + 273.16
    Tavg4 = (Tmax_K**4 + Tmin_K**4) / 2.0

    Rso = clear_sky_radiation(Ra, elevation_m)
    Rs_Rso = Rs / Rso if Rso > 0 else 0.0
    Rs_Rso = max(0.0, min(Rs_Rso, 1.5))  # limit ratio to reasonable range

    Rnl = sigma * Tavg4 * (0.34 - 0.14 * math.sqrt(max(ea, 0.0))) * (1.35 * Rs_Rso - 0.35)

    return Rns - Rnl

def compute_daily_et0_row(row, elevation_m=10.0, latitude_deg=6.94, day_of_year=1):
    """
    Compute ET0 for a single day using the FAO Penman-Monteith formula (daily).

    Expects a row with:
      temperature_2m_max, temperature_2m_min,
      relative_humidity_2m_max, relative_humidity_2m_min,
      wind_speed_10m_max, shortwave_radiation_sum (MJ/m²/day)
    """
    Tmax = row["temperature_2m_max"]
    Tmin = row["temperature_2m_min"]
    RHmax = row["relative_humidity_2m_max"]
    RHmin = row["relative_humidity_2m_min"]
    u10 = row["wind_speed_10m_max"]           # m/s at 10 m (Open-Meteo daily max or mean) [web:73]
    Rs = row["shortwave_radiation_sum"]       # MJ/m² per day (Open-Meteo) [web:74]

    Tmean = (Tmax + Tmin) / 2.0

    # Saturation vapour pressures
    es_max = saturation_vapor_pressure(Tmax)
    es_min = saturation_vapor_pressure(Tmin)
    es = (es_max + es_min) / 2.0

    # Actual vapour pressure
    ea = actual_vapor_pressure(RHmin, RHmax, es_min, es_max)

    # Slope of SVP curve
    delta = slope_svp_curve(Tmean)

    # Psychrometric constant
    gamma = psychrometric_constant(elevation_m)

    # Radiation: Ra and full FAO-56 net radiation
    Ra = extraterrestrial_radiation(day_of_year, latitude_deg)
    Rn = net_radiation_fao(Rs, Tmax, Tmin, ea, Ra, elevation_m)

    # Soil heat flux for daily step
    G = 0.0

    # Convert wind from 10 m to 2 m
    u2 = wind_speed_10m_to_2m(u10)

    # FAO Penman-Monteith equation (daily ET0, mm/day)
    numerator = 0.408 * delta * (Rn - G) + gamma * (900.0 / (Tmean + 273.0)) * u2 * (es - ea)
    denominator = delta + gamma * (1.0 + 0.34 * u2)
    ET0 = numerator / max(denominator, 1e-6)

    return ET0


final

In [1]:
def saturation_vapor_pressure(T):
    """Saturation vapour pressure e_s(T) in kPa for temperature T in Celsius."""
    return 0.6108 * math.exp((17.27 * T) / (T + 237.3))

def slope_svp_curve(T):
    """Slope of saturation vapour pressure curve Delta in kPa/°C."""
    es_T = saturation_vapor_pressure(T)
    return 4098.0 * es_T / ((T + 237.3) ** 2)

def actual_vapor_pressure(RHmin, RHmax, es_min, es_max):
    """Actual vapour pressure ea in kPa."""
    ea = (RHmax / 100.0) * es_min + (RHmin / 100.0) * es_max
    return ea / 2.0

def psychrometric_constant(elevation_m):
    """Psychrometric constant gamma in kPa/°C."""
    P = 101.3 * ((293.0 - 0.0065 * elevation_m) / 293.0) ** 5.26
    gamma = 0.000665 * P
    return gamma

def wind_speed_10m_to_2m(u10):
    """Convert wind speed from 10 m to 2 m (FAO‑56)."""
    z = 10.0
    return u10 * 4.87 / math.log(67.8 * z - 5.42)

def extraterrestrial_radiation(day_of_year, latitude_deg):
    """Extraterrestrial radiation Ra (MJ/m²/day)."""
    phi = math.radians(latitude_deg)
    dr = 1 + 0.033 * math.cos((2 * math.pi / 365) * day_of_year)
    delta = 0.409 * math.sin((2 * math.pi / 365) * day_of_year - 1.39)
    ws = math.acos(-math.tan(phi) * math.tan(delta))
    Gsc = 0.0820  # MJ m-2 min-1

    Ra = (24 * 60 / math.pi) * Gsc * dr * (
        ws * math.sin(phi) * math.sin(delta) +
        math.cos(phi) * math.cos(delta) * math.sin(ws)
    )
    return Ra

def clear_sky_radiation(Ra, elevation_m):
    """Clear-sky solar radiation Rso (MJ/m²/day)."""
    return (0.75 + 2e-5 * elevation_m) * Ra

def net_radiation_fao(Rs, Tmax, Tmin, ea, Ra, elevation_m):
    """Net radiation Rn (MJ/m²/day) using FAO‑56."""
    albedo = 0.23
    Rns = (1.0 - albedo) * Rs

    sigma = 4.903e-9  # MJ K^-4 m^-2 day^-1
    Tmax_K = Tmax + 273.16
    Tmin_K = Tmin + 273.16
    Tavg4 = (Tmax_K**4 + Tmin_K**4) / 2.0

    Rso = clear_sky_radiation(Ra, elevation_m)
    Rs_Rso = Rs / Rso if Rso > 0 else 0.0
    Rs_Rso = max(0.0, min(Rs_Rso, 1.5))

    Rnl = sigma * Tavg4 * (0.34 - 0.14 * math.sqrt(max(ea, 0.0))) * (1.35 * Rs_Rso - 0.35)

    return Rns - Rnl

def compute_daily_et0(Tmax, Tmin, RHmax, RHmin, u10_mean_kmh, Rs,
                      elevation_m, latitude_deg, day_of_year):
    """Daily FAO‑56 Penman–Monteith ET0 (mm/day)."""
    Tmean = (Tmax + Tmin) / 2.0

    es_max = saturation_vapor_pressure(Tmax)
    es_min = saturation_vapor_pressure(Tmin)
    es = (es_max + es_min) / 2.0

    ea = actual_vapor_pressure(RHmin, RHmax, es_min, es_max)

    delta = slope_svp_curve(Tmean)
    gamma = psychrometric_constant(elevation_m)

    Ra = extraterrestrial_radiation(day_of_year, latitude_deg)
    Rn = net_radiation_fao(Rs, Tmax, Tmin, ea, Ra, elevation_m)
    
    # Convert km/h → m/s
    u10_mean = u10_mean_kmh / 3.6
    
    G = 0.0
    u2 = wind_speed_10m_to_2m(u10_mean)

    numerator = 0.408 * delta * (Rn - G) + gamma * (900.0 / (Tmean + 273.0)) * u2 * (es - ea)
    denominator = delta + gamma * (1.0 + 0.34 * u2)
    return numerator / max(denominator, 1e-6)


In [2]:
import pandas as pd
from datetime import date
import requests
import math

In [3]:
# ============================================================
# 2. Fetch today's daily data for Colombo from Open-Meteo
# ============================================================

LATITUDE = 6.9271
LONGITUDE = 79.8612
ELEVATION_M = 7          # ~average elevation for Colombo [web:63][web:65]
today = date.today().isoformat()
day_of_year = date.today().timetuple().tm_yday

open_meteo_url = "https://archive-api.open-meteo.com/v1/era5"

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": today,
    "end_date": today,
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "relative_humidity_2m_max",
        "relative_humidity_2m_min",
        "wind_speed_10m_mean",
        "shortwave_radiation_sum",
    ],
    "timezone": "Asia/Colombo",
    # wind_speed_unit left as default (km/h) – we convert in code [web:9][web:37]
}

resp = requests.get(open_meteo_url, params=params)
resp.raise_for_status()
daily = resp.json()["daily"]

df = pd.DataFrame(daily)
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time").sort_index()

# Take the first (and only) row = today
row = df.iloc[0]

Tmax = row["temperature_2m_max"]
Tmin = row["temperature_2m_min"]
RHmax = row["relative_humidity_2m_max"]
RHmin = row["relative_humidity_2m_min"]
u10_mean_kmh = row["wind_speed_10m_mean"]

Rs = row["shortwave_radiation_sum"]

# ============================================================
# 3. Compute ET0 and print
# ============================================================

ET0_today = compute_daily_et0(
    Tmax=Tmax,
    Tmin=Tmin,
    RHmax=RHmax,
    RHmin=RHmin,
    u10_mean_kmh=u10_mean_kmh,
    Rs=Rs,
    elevation_m=ELEVATION_M,
    latitude_deg=LATITUDE,
    day_of_year=day_of_year,
)

print(f"Date: {today}")
print("Location: Colombo, Sri Lanka")
print(f"Daily ET0 (FAO-56 Penman-Monteith): {ET0_today:.2f} mm/day")

## Single code cell

In [5]:
import math
import requests
import pandas as pd
from datetime import date

# --- FAO-56 Helper Functions ---

def saturation_vapor_pressure(T):
    return 0.6108 * math.exp((17.27 * T) / (T + 237.3))

def slope_svp_curve(T):
    return 4098.0 * saturation_vapor_pressure(T) / ((T + 237.3) ** 2)

def actual_vapor_pressure(RHmin, RHmax, es_min, es_max):
    return ((RHmax / 100.0) * es_min + (RHmin / 100.0) * es_max) / 2.0

def psychrometric_constant(elevation_m):
    P = 101.3 * ((293.0 - 0.0065 * elevation_m) / 293.0) ** 5.26
    return 0.000665 * P

def wind_speed_10m_to_2m(u10):
    return u10 * 4.87 / math.log(67.8 * 10.0 - 5.42)

def extraterrestrial_radiation(day_of_year, latitude_deg):
    phi = math.radians(latitude_deg)
    dr = 1 + 0.033 * math.cos((2 * math.pi / 365) * day_of_year)
    delta = 0.409 * math.sin((2 * math.pi / 365) * day_of_year - 1.39)
    ws = math.acos(-math.tan(phi) * math.tan(delta))
    return (24 * 60 / math.pi) * 0.0820 * dr * (
        ws * math.sin(phi) * math.sin(delta) +
        math.cos(phi) * math.cos(delta) * math.sin(ws)
    )

def net_radiation_fao(Rs, Tmax, Tmin, ea, Ra, elevation_m):
    Rns = (1.0 - 0.23) * Rs
    Rso = (0.75 + 2e-5 * elevation_m) * Ra
    Rs_Rso = max(0.0, min(Rs / Rso if Rso > 0 else 0.0, 1.5))
    Tavg4 = ((Tmax + 273.16) ** 4 + (Tmin + 273.16) ** 4) / 2.0
    Rnl = 4.903e-9 * Tavg4 * (0.34 - 0.14 * math.sqrt(max(ea, 0.0))) * (1.35 * Rs_Rso - 0.35)
    return Rns - Rnl

def compute_daily_eto(Tmax, Tmin, RHmax, RHmin, u10_kmh, Rs, elevation_m, latitude_deg, day_of_year):
    Tmean = (Tmax + Tmin) / 2.0
    es = (saturation_vapor_pressure(Tmax) + saturation_vapor_pressure(Tmin)) / 2.0
    ea = actual_vapor_pressure(RHmin, RHmax, saturation_vapor_pressure(Tmin), saturation_vapor_pressure(Tmax))
    delta = slope_svp_curve(Tmean)
    gamma = psychrometric_constant(elevation_m)
    Ra = extraterrestrial_radiation(day_of_year, latitude_deg)
    Rn = net_radiation_fao(Rs, Tmax, Tmin, ea, Ra, elevation_m)
    u2 = wind_speed_10m_to_2m(u10_kmh / 3.6)
    numerator = 0.408 * delta * Rn + gamma * (900.0 / (Tmean + 273.0)) * u2 * (es - ea)
    denominator = delta + gamma * (1.0 + 0.34 * u2)
    return numerator / max(denominator, 1e-6)

# --- Fetch Today's Data from Open-Meteo ---

LATITUDE   = 6.9271
LONGITUDE  = 79.8612
ELEVATION_M = 7
today      = date.today().isoformat()
doy        = date.today().timetuple().tm_yday

params = {
    "latitude": LATITUDE, "longitude": LONGITUDE,
    "start_date": today, "end_date": today,
    "daily": ["temperature_2m_max", "temperature_2m_min",
              "relative_humidity_2m_max", "relative_humidity_2m_min",
              "wind_speed_10m_mean", "shortwave_radiation_sum"],
    "timezone": "Asia/Colombo"
}

row = pd.DataFrame(requests.get("https://archive-api.open-meteo.com/v1/era5", params=params)
                   .raise_for_status() or
                   requests.get("https://archive-api.open-meteo.com/v1/era5", params=params)
                   .json()["daily"]).iloc[0]

# --- Compute and Print ETo ---

eto = compute_daily_eto(
    Tmax=row["temperature_2m_max"], Tmin=row["temperature_2m_min"],
    RHmax=row["relative_humidity_2m_max"], RHmin=row["relative_humidity_2m_min"],
    u10_kmh=row["wind_speed_10m_mean"], Rs=row["shortwave_radiation_sum"],
    elevation_m=ELEVATION_M, latitude_deg=LATITUDE, day_of_year=doy
)

print(f"{eto:.2f} mm/day")


In [6]:
import math
import requests
from datetime import date

# --- FAO-56 Helper Functions ---

def saturation_vapor_pressure(T):
    return 0.6108 * math.exp((17.27 * T) / (T + 237.3))

def slope_svp_curve(T):
    return 4098.0 * saturation_vapor_pressure(T) / ((T + 237.3) ** 2)

def actual_vapor_pressure(RHmin, RHmax, es_min, es_max):
    return ((RHmax / 100.0) * es_min + (RHmin / 100.0) * es_max) / 2.0

def psychrometric_constant(elevation_m):
    P = 101.3 * ((293.0 - 0.0065 * elevation_m) / 293.0) ** 5.26
    return 0.000665 * P

def wind_speed_10m_to_2m(u10):
    return u10 * 4.87 / math.log(67.8 * 10.0 - 5.42)

def extraterrestrial_radiation(day_of_year, latitude_deg):
    phi = math.radians(latitude_deg)
    dr = 1 + 0.033 * math.cos((2 * math.pi / 365) * day_of_year)
    delta = 0.409 * math.sin((2 * math.pi / 365) * day_of_year - 1.39)
    ws = math.acos(-math.tan(phi) * math.tan(delta))
    return (24 * 60 / math.pi) * 0.0820 * dr * (
        ws * math.sin(phi) * math.sin(delta) +
        math.cos(phi) * math.cos(delta) * math.sin(ws)
    )

def net_radiation_fao(Rs, Tmax, Tmin, ea, Ra, elevation_m):
    Rns = (1.0 - 0.23) * Rs
    Rso = (0.75 + 2e-5 * elevation_m) * Ra
    Rs_Rso = max(0.0, min(Rs / Rso if Rso > 0 else 0.0, 1.5))
    Tavg4 = ((Tmax + 273.16) ** 4 + (Tmin + 273.16) ** 4) / 2.0
    Rnl = 4.903e-9 * Tavg4 * (0.34 - 0.14 * math.sqrt(max(ea, 0.0))) * (1.35 * Rs_Rso - 0.35)
    return Rns - Rnl

def compute_daily_eto(Tmax, Tmin, RHmax, RHmin, u10_kmh, Rs, elevation_m, latitude_deg, day_of_year):
    Tmean = (Tmax + Tmin) / 2.0
    es = (saturation_vapor_pressure(Tmax) + saturation_vapor_pressure(Tmin)) / 2.0
    ea = actual_vapor_pressure(RHmin, RHmax, saturation_vapor_pressure(Tmin), saturation_vapor_pressure(Tmax))
    delta = slope_svp_curve(Tmean)
    gamma = psychrometric_constant(elevation_m)
    Ra = extraterrestrial_radiation(day_of_year, latitude_deg)
    Rn = net_radiation_fao(Rs, Tmax, Tmin, ea, Ra, elevation_m)
    u2 = wind_speed_10m_to_2m(u10_kmh / 3.6)
    numerator = 0.408 * delta * Rn + gamma * (900.0 / (Tmean + 273.0)) * u2 * (es - ea)
    denominator = delta + gamma * (1.0 + 0.34 * u2)
    return numerator / max(denominator, 1e-6)

# --- Fetch Today's Data ---
LATITUDE    = 6.9271
LONGITUDE   = 79.8612
ELEVATION_M = 7
today = date.today().isoformat()
doy   = date.today().timetuple().tm_yday

params = {
    "latitude": LATITUDE, "longitude": LONGITUDE,
    "start_date": today, "end_date": today,
    "daily": ["temperature_2m_max", "temperature_2m_min",
              "relative_humidity_2m_max", "relative_humidity_2m_min",
              "wind_speed_10m_mean", "shortwave_radiation_sum"],
    "timezone": "Asia/Colombo"
}

response = requests.get("https://api.open-meteo.com/v1/forecast", params=params)
response.raise_for_status()
data = response.json()["daily"]

# --- Compute ETo ---
eto = compute_daily_eto(
    Tmax        = data["temperature_2m_max"][0],
    Tmin        = data["temperature_2m_min"][0],
    RHmax       = data["relative_humidity_2m_max"][0],
    RHmin       = data["relative_humidity_2m_min"][0],
    u10_kmh     = data["wind_speed_10m_mean"][0],
    Rs          = data["shortwave_radiation_sum"][0],
    elevation_m = ELEVATION_M,
    latitude_deg= LATITUDE,
    day_of_year = doy
)

print(f"{eto:.2f} mm/day")


In [11]:
# ============================================================
# 2. Fetch selected day's daily data for Colombo from Open-Meteo then calculate ETo using our code
# ============================================================

LATITUDE = 6.9271
LONGITUDE = 79.8612
ELEVATION_M = 5.0          # ~average elevation for Colombo

# --- choose the date you want here ---
from datetime import datetime

TARGET_DATE = "2026-02-18"   # <- change this to any date you want
target_dt = datetime.strptime(TARGET_DATE, "%Y-%m-%d")
day_of_year = target_dt.timetuple().tm_yday
# -------------------------------------

open_meteo_url = "https://archive-api.open-meteo.com/v1/era5"

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": TARGET_DATE,
    "end_date": TARGET_DATE,
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "relative_humidity_2m_max",
        "relative_humidity_2m_min",
        "wind_speed_10m_mean",
        "shortwave_radiation_sum",
    ],
    "timezone": "Asia/Colombo",
    # wind_speed_unit left as default (km/h) – we convert in code
}

resp = requests.get(open_meteo_url, params=params)
resp.raise_for_status()
daily = resp.json()["daily"]

df = pd.DataFrame(daily)
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time").sort_index()

# Take the first (and only) row = TARGET_DATE
row = df.iloc[0]

Tmax = row["temperature_2m_max"]
Tmin = row["temperature_2m_min"]
RHmax = row["relative_humidity_2m_max"]
RHmin = row["relative_humidity_2m_min"]
u10_mean_kmh = row["wind_speed_10m_mean"]
Rs = row["shortwave_radiation_sum"]

# ============================================================
# 3. Compute ET0 and print for the selected date
# ============================================================

ET0_selected = compute_daily_et0(
    Tmax=Tmax,
    Tmin=Tmin,
    RHmax=RHmax,
    RHmin=RHmin,
    u10_mean_kmh=u10_mean_kmh,
    Rs=Rs,
    elevation_m=ELEVATION_M,
    latitude_deg=LATITUDE,
    day_of_year=day_of_year,
)

print(f"Date: {TARGET_DATE}")
print("Location: Colombo, Sri Lanka")
print(f"Daily ET0 (FAO-56 Penman-Monteith): {ET0_selected:.2f} mm/day")


In [14]:
import requests
from datetime import datetime

# Colombo coordinates
lat = 6.9271
lon = 79.8612

# --- choose your custom past date here (YYYY-MM-DD) ---
CUSTOM_DATE = "2026-02-18"
# ------------------------------------------------------

# Parse date and get day of year (not strictly needed if using ET0 from API,
# but useful if you later want to compute ET0 yourself)
dt = datetime.strptime(CUSTOM_DATE, "%Y-%m-%d")
doy = dt.timetuple().tm_yday

# Historical ERA5 endpoint (not forecast)
url = "https://archive-api.open-meteo.com/v1/era5"

params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": CUSTOM_DATE,
    "end_date": CUSTOM_DATE,
    "daily": [
        "et0_fao_evapotranspiration"
    ],
    "timezone": "Asia/Colombo",
}

resp = requests.get(url, params=params)
resp.raise_for_status()
data = resp.json()

dates = data["daily"]["time"]
et0 = data["daily"]["et0_fao_evapotranspiration"]

if CUSTOM_DATE in dates and len(et0) > 0:
    idx = dates.index(CUSTOM_DATE)
    print(f"ET0 (FAO-56) in Colombo on {CUSTOM_DATE}: {et0[idx]} mm/day")
else:
    print(f"ET0 for {CUSTOM_DATE} not found in historical API response")


In [13]:
import requests
from datetime import datetime

# Colombo coordinates
lat = 6.9271
lon = 79.8612

# --- choose your custom date here (YYYY-MM-DD) ---
CUSTOM_DATE = "2026-02-18"
# -------------------------------------------------

url = (
    f"https://api.open-meteo.com/v1/forecast?"
    f"latitude={lat}&longitude={lon}"
    f"&daily=et0_fao_evapotranspiration"
    f"&timezone=Asia%2FColombo"
)

response = requests.get(url)
response.raise_for_status()
data = response.json()

dates = data["daily"]["time"]
et0 = data["daily"]["et0_fao_evapotranspiration"]

if CUSTOM_DATE in dates:
    idx = dates.index(CUSTOM_DATE)
    print(f"ET0 (FAO-56) in Colombo on {CUSTOM_DATE}: {et0[idx]} mm/day")
else:
    print(f"ET0 for {CUSTOM_DATE} not found in API response")
